In [ ]:
from langgraph.graph import StateGraph, START, END, add_messages
from typing import TypedDict, Annotated, List
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import AzureChatOpenAI
from langchain_community.tools import TavilySearchResults
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv,find_dotenv
import os

In [ ]:
load_dotenv(find_dotenv(),override=True)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version="2025-01-01-preview"

memory=MemorySaver()
search_tool=TavilySearchResults(max_results=2)
tools=[search_tool]
llm_with_tools = AzureChatOpenAI(azure_endpoint=endpoint,api_key=subscription_key, api_version=api_version).bind_tools(tools)

class BasicState(TypedDict): 
    messages: Annotated[List, add_messages]

In [2]:
def model(state: BasicState): 
    return { "messages": [llm_with_tools.invoke(state["messages"])]  }

def tools_router(state: BasicState): 
    last_message = state["messages"][-1]
    if(hasattr(last_message, "tool_calls") and len(last_message.tool_calls) > 0):
        return "tools"
    else: 
        return END


graph = StateGraph(BasicState)
graph.add_node(model, "model")
graph.add_node("tools", ToolNode(tools=tools))

graph.set_entry_point("model")
graph.add_conditional_edges("model", tools_router)

graph.add_edge("tools", "model")

app = graph.compile(checkpointer=memory, interrupt_before=["tools"])

NameError: name 'BasicState' is not defined

In [ ]:
config = {"configurable": {
    "thread_id": 1
}}

events = app.stream({
    "messages": [HumanMessage(content="What is the current weather in Chennai?")]
}, config=config, stream_mode="values")

for event in events:
    event["messages"][-1].pretty_print()